# Dataset inspection
Run after uploading the real dataset. These descriptive summaries are not model-selection evidence.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "backend/app").is_dir())
path = root / "data/raw/MdrDB_Full_Scraped.csv"
if not path.is_file() or path.stat().st_size == 0:
    raise FileNotFoundError(f"Upload the actual dataset to {path}")
df = pd.read_csv(path, low_memory=False)
if "DDG.EXP" not in df:
    raise ValueError("The dataset must contain DDG.EXP")
df.shape


In [ ]:
target = pd.to_numeric(df["DDG.EXP"], errors="coerce")
pd.Series({"rows": len(df), "columns": len(df.columns), "missing_target": int(target.isna().sum()), "nonfinite_target": int((target.notna() & ~np.isfinite(target)).sum()), "duplicate_rows": int(df.duplicated().sum())})


In [ ]:
families = {name: [c for c in df if c.startswith(name + ".")] for name in ["LIG", "MUT", "ENV", "PLIP", "P2", "P2dl", "P2dp"]}
families["VINA"] = [c for c in df if c == "vina" or (c.startswith("vina") and c[4:].isdigit())]
pd.Series({k: len(v) for k, v in families.items()}, name="Feature count")


In [ ]:
features = [c for columns in families.values() for c in columns]
numeric = df[features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
summary = pd.DataFrame({"missing_fraction": numeric.isna().mean(), "unique_values": numeric.nunique()})
output = root / "results" / "data_analysis"
output.mkdir(parents=True, exist_ok=True)
summary.to_csv(output / "feature_quality.csv")
summary.sort_values("missing_fraction", ascending=False).head(25)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
target.replace([np.inf, -np.inf], np.nan).dropna().hist(bins=50, ax=ax)
ax.set(xlabel="DDG.EXP (dataset units)", ylabel="Rows", title="Experimental target distribution")
fig.tight_layout()
fig.savefig(output / "target_distribution.png", dpi=150)
plt.show()
